In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, norm

In [9]:

# ============================================================
# CONFIGURATION
# ============================================================

SO_DIR = Path("Stack Overflow")
GH_DIR = Path("GitHub")

OUTPUT_DIR = Path("RQ2/cross_platform_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_PATTERN = "labeled_T*.csv"

QUESTION_TYPES = ["how", "why", "what", "other"]

# After the first run, complete the generated resolution template
# and set this path to the completed file.
MANUAL_RESOLUTION_FILE = Path("manual_duplicate_resolutions.csv")

In [3]:
def normalize_question_type(value):
    """Normalize variations of the four question-type labels."""
    if pd.isna(value):
        return np.nan

    value = (
        str(value)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )

    mapping = {
        "how": "how",
        "how type": "how",
        "why": "why",
        "why type": "why",
        "what": "what",
        "what type": "what",
        "other": "other",
        "others": "other",
        "other type": "other",
    }

    return mapping.get(value, np.nan)


def normalize_id(value):
    """Convert numeric IDs such as 12345.0 to stable strings."""
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text.endswith(".0"):
        try:
            return str(int(float(text)))
        except ValueError:
            pass

    return text


def detect_repository_column(df):
    """Detect a repository-name column when one exists."""
    candidates = [
        "repository",
        "repo",
        "repo_name",
        "repository_name",
        "full_name",
        "repo_full_name",
    ]

    return next(
        (column for column in candidates if column in df.columns),
        None,
    )


In [4]:
def load_platform(folder, platform):
    """Load all topic files for one platform."""

    files = sorted(folder.rglob(FILE_PATTERN))

    if not files:
        raise FileNotFoundError(
            f"No files matching {FILE_PATTERN!r} were found in "
            f"{folder.resolve()}"
        )

    frames = []

    for file in files:
        df = pd.read_csv(file)

        id_column = "Id" if platform == "Stack Overflow" else "id"

        required_columns = {
            id_column,
            "Topic",
            "question_type",
        }

        missing_columns = required_columns.difference(df.columns)

        if missing_columns:
            raise ValueError(
                f"{file} is missing required columns: "
                f"{sorted(missing_columns)}"
            )

        repository_column = detect_repository_column(df)

        if "Title" in df.columns:
            title_column = "Title"
        elif "title" in df.columns:
            title_column = "title"
        else:
            title_column = None

        output = pd.DataFrame(
            {
                "platform": platform,
                "artifact_id": df[id_column].map(normalize_id),
                "topic": df["Topic"],
                "question_type": df["question_type"].map(
                    normalize_question_type
                ),
                "source_file": file.name,
                "title": (
                    df[title_column].fillna("").astype(str)
                    if title_column
                    else ""
                ),
                "document": (
                    df["Document"].fillna("").astype(str)
                    if "Document" in df.columns
                    else ""
                ),
                "repository": (
                    df[repository_column]
                    .fillna("")
                    .astype(str)
                    .str.strip()
                    if repository_column
                    else ""
                ),
            }
        )

        frames.append(output)

    data = pd.concat(frames, ignore_index=True)

    invalid_rows = data[
        data["artifact_id"].isna()
        | data["question_type"].isna()
    ]

    if not invalid_rows.empty:
        platform_name = platform.lower().replace(" ", "_")

        invalid_rows.to_csv(
            OUTPUT_DIR / f"{platform_name}_invalid_rows.csv",
            index=False,
        )

        print(
            f"{platform}: excluded {len(invalid_rows)} rows "
            "with missing IDs or invalid question-type labels."
        )

    data = data.dropna(
        subset=["artifact_id", "question_type"]
    ).copy()

    # If repository information exists, use repository + ID.
    # Otherwise, use the ID alone.
    if data["repository"].astype(bool).any():
        data["artifact_key"] = (
            data["repository"] + "::" + data["artifact_id"]
        )
    else:
        data["artifact_key"] = data["artifact_id"]

    print(
        f"{platform}: loaded {len(data):,} rows from "
        f"{len(files)} files."
    )

    return data


In [5]:
# ============================================================
# DUPLICATE AUDIT AND RESOLUTION
# ============================================================

def resolve_duplicates(data):
    """
    Identify repeated artifacts.

    Duplicates with the same label are reduced to one artifact.
    Duplicates with conflicting labels require manual resolution.
    """

    duplicate_mask = data.duplicated(
        ["platform", "artifact_key"],
        keep=False,
    )

    duplicates = data.loc[duplicate_mask].copy()

    duplicates.to_csv(
        OUTPUT_DIR / "duplicate_artifacts_audit.csv",
        index=False,
    )

    if duplicates.empty:
        print("No duplicate artifacts were found.")
        return data

    duplicate_label_counts = (
        duplicates
        .groupby(["platform", "artifact_key"])["question_type"]
        .nunique()
    )

    conflicting_keys = duplicate_label_counts[
        duplicate_label_counts > 1
    ].index

    if len(conflicting_keys) > 0:
        conflicts = (
            duplicates
            .set_index(["platform", "artifact_key"])
            .loc[conflicting_keys]
            .reset_index()
        )
    else:
        conflicts = duplicates.iloc[0:0].copy()

    conflicts.to_csv(
        OUTPUT_DIR / "conflicting_duplicate_labels.csv",
        index=False,
    )

    if not conflicts.empty:
        resolution_template = (
            conflicts[
                ["platform", "artifact_key", "title"]
            ]
            .drop_duplicates(
                ["platform", "artifact_key"]
            )
            .assign(question_type="")
        )

        template_path = (
            OUTPUT_DIR
            / "manual_duplicate_resolutions_template.csv"
        )

        resolution_template.to_csv(
            template_path,
            index=False,
        )

        if not MANUAL_RESOLUTION_FILE.exists():
            raise RuntimeError(
                f"Found {len(resolution_template):,} duplicated "
                "artifacts with conflicting question-type labels.\n"
                f"Review: "
                f"{OUTPUT_DIR / 'conflicting_duplicate_labels.csv'}\n"
                f"Complete: {template_path}\n"
                "Then set MANUAL_RESOLUTION_FILE to the completed file "
                "and rerun the analysis."
            )

        resolutions = pd.read_csv(
            MANUAL_RESOLUTION_FILE
        )

        required_resolution_columns = {
            "platform",
            "artifact_key",
            "question_type",
        }

        missing_columns = (
            required_resolution_columns
            .difference(resolutions.columns)
        )

        if missing_columns:
            raise ValueError(
                "The manual-resolution file is missing columns: "
                f"{sorted(missing_columns)}"
            )

        resolutions["artifact_key"] = (
            resolutions["artifact_key"]
            .astype(str)
            .str.strip()
        )

        resolutions["question_type"] = (
            resolutions["question_type"]
            .map(normalize_question_type)
        )

        if resolutions["question_type"].isna().any():
            raise ValueError(
                "The manual-resolution file contains missing or "
                "invalid question-type labels."
            )

        if resolutions.duplicated(
            ["platform", "artifact_key"]
        ).any():
            raise ValueError(
                "The manual-resolution file contains duplicate keys."
            )

        data = data.merge(
            resolutions[
                [
                    "platform",
                    "artifact_key",
                    "question_type",
                ]
            ].rename(
                columns={
                    "question_type":
                        "manual_question_type"
                }
            ),
            on=["platform", "artifact_key"],
            how="left",
        )

        data["question_type"] = (
            data["manual_question_type"]
            .fillna(data["question_type"])
        )

        data = data.drop(
            columns="manual_question_type"
        )

        remaining_conflicts = (
            data
            .groupby(
                ["platform", "artifact_key"]
            )["question_type"]
            .nunique()
            .gt(1)
        )

        if remaining_conflicts.any():
            raise RuntimeError(
                "Some conflicting duplicate labels remain unresolved."
            )

    # Keep one representation per artifact.
    # When an artifact appears more than once, retain the row
    # containing the longest document text.
    data["document_length"] = (
        data["document"].str.len()
    )

    data = (
        data
        .sort_values(
            [
                "platform",
                "artifact_key",
                "document_length",
            ],
            ascending=[True, True, False],
        )
        .drop_duplicates(
            ["platform", "artifact_key"],
            keep="first",
        )
        .drop(columns="document_length")
    )

    print(
        f"Final dataset after duplicate resolution: "
        f"{len(data):,} unique artifacts."
    )

    return data

In [6]:
# ============================================================
# STATISTICAL FUNCTIONS
# ============================================================

def adjusted_standardized_residuals(
    observed,
    expected,
):
    """
    Calculate adjusted standardized residuals.

    Positive values indicate overrepresentation.
    Negative values indicate underrepresentation.
    """

    observed = np.asarray(
        observed,
        dtype=float,
    )

    expected = np.asarray(
        expected,
        dtype=float,
    )

    total = observed.sum()

    row_proportions = (
        observed.sum(axis=1, keepdims=True)
        / total
    )

    column_proportions = (
        observed.sum(axis=0, keepdims=True)
        / total
    )

    denominator = np.sqrt(
        expected
        * (1 - row_proportions)
        * (1 - column_proportions)
    )

    return (observed - expected) / denominator


def calculate_odds_ratio(a, b, c, d):
    """
    Table orientation:

                       Selected type    Other types
    Stack Overflow          a               b
    GitHub                  c               d

    OR > 1 indicates stronger association with Stack Overflow.
    OR < 1 indicates stronger association with GitHub.
    """

    cells = np.array(
        [a, b, c, d],
        dtype=float,
    )

    # Haldane–Anscombe correction for zero cells.
    if np.any(cells == 0):
        cells += 0.5

    a, b, c, d = cells

    odds_ratio = (a * d) / (b * c)

    standard_error = np.sqrt(
        1 / a
        + 1 / b
        + 1 / c
        + 1 / d
    )

    log_odds_ratio = np.log(odds_ratio)

    lower_ci = np.exp(
        log_odds_ratio
        - 1.96 * standard_error
    )

    upper_ci = np.exp(
        log_odds_ratio
        + 1.96 * standard_error
    )

    z_value = (
        log_odds_ratio
        / standard_error
    )

    p_value = (
        2
        * norm.sf(abs(z_value))
    )

    return (
        odds_ratio,
        lower_ci,
        upper_ci,
        p_value,
    )


In [7]:
def holm_adjustment(p_values):
    """Apply Holm correction to multiple p-values."""

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    order = np.argsort(p_values)
    ordered_p_values = p_values[order]

    adjusted_ordered = np.empty_like(
        ordered_p_values
    )

    running_maximum = 0.0
    number_of_tests = len(p_values)

    for index, p_value in enumerate(
        ordered_p_values
    ):
        adjusted_value = (
            number_of_tests - index
        ) * p_value

        running_maximum = max(
            running_maximum,
            adjusted_value,
        )

        adjusted_ordered[index] = min(
            running_maximum,
            1.0,
        )

    adjusted = np.empty_like(
        adjusted_ordered
    )

    adjusted[order] = adjusted_ordered

    return adjusted


In [11]:
# ============================================================
# LOAD AND CLEAN ALL FILES
# ============================================================

so_data = load_platform(
    SO_DIR,
    "Stack Overflow",
)

github_data = load_platform(
    GH_DIR,
    "GitHub",
)

combined_data = pd.concat(
    [so_data, github_data],
    ignore_index=True,
)

conflicting_df = pd.read_csv(OUTPUT_DIR / "conflicting_duplicate_labels.csv")
print(f"Found {len(conflicting_df)} conflicting duplicates")
print(conflicting_df.head(20))

combined_data.to_csv(
    OUTPUT_DIR / "combined_clean_artifacts.csv",
    index=False,
)


Stack Overflow: loaded 495 rows from 9 files.
GitHub: excluded 3 rows with missing IDs or invalid question-type labels.
GitHub: loaded 9,113 rows from 13 files.
Found 3742 conflicting duplicates
   platform  artifact_key  artifact_id  topic question_type      source_file  \
0    GitHub    1002104246   1002104246     -1           why  labeled_T-1.csv   
1    GitHub    1002104246   1002104246     -1          what  labeled_T-1.csv   
2    GitHub    1004921341   1004921341     -1           why  labeled_T-1.csv   
3    GitHub    1004921341   1004921341     -1         other  labeled_T-1.csv   
4    GitHub    1005049002   1005049002     -1          what  labeled_T-1.csv   
5    GitHub    1005049002   1005049002     -1           how  labeled_T-1.csv   
6    GitHub    1005051076   1005051076     -1           why  labeled_T-1.csv   
7    GitHub    1005051076   1005051076     -1          what  labeled_T-1.csv   
8    GitHub    1006287750   1006287750     -1           why  labeled_T-1.csv   
9    

In [12]:


















# ============================================================
# CONTINGENCY TABLE
# ============================================================

counts = pd.crosstab(
    combined_data["platform"],
    combined_data["question_type"],
)

counts = counts.reindex(
    index=["Stack Overflow", "GitHub"],
    columns=QUESTION_TYPES,
    fill_value=0,
)

percentages = (
    counts
    .div(counts.sum(axis=1), axis=0)
    * 100
)

print("\nObserved counts")
print(counts)

print("\nPlatform percentages")
print(percentages.round(2))

counts.to_csv(
    OUTPUT_DIR / "question_type_counts.csv"
)

percentages.to_csv(
    OUTPUT_DIR / "question_type_percentages.csv"
)


# ============================================================
# CHI-SQUARE AND CRAMÉR'S V
# ============================================================

(
    chi_square,
    chi_square_p,
    degrees_of_freedom,
    expected_counts,
) = chi2_contingency(
    counts,
    correction=False,
)

sample_size = int(
    counts.to_numpy().sum()
)

cramers_v = np.sqrt(
    chi_square
    / (
        sample_size
        * min(
            counts.shape[0] - 1,
            counts.shape[1] - 1,
        )
    )
)

print("\nOverall cross-platform test")
print(f"Chi-square: {chi_square:.4f}")
print(f"Degrees of freedom: {degrees_of_freedom}")
print(f"p-value: {chi_square_p:.8g}")
print(f"Cramér's V: {cramers_v:.4f}")


# ============================================================
# ADJUSTED STANDARDIZED RESIDUALS
# ============================================================

residuals = pd.DataFrame(
    adjusted_standardized_residuals(
        counts.to_numpy(),
        expected_counts,
    ),
    index=counts.index,
    columns=counts.columns,
)

print("\nAdjusted standardized residuals")
print(residuals.round(3))

residuals.to_csv(
    OUTPUT_DIR
    / "adjusted_standardized_residuals.csv"
)


# ============================================================
# ODDS RATIOS AND CONFIDENCE INTERVALS
# ============================================================

so_total = counts.loc[
    "Stack Overflow"
].sum()

github_total = counts.loc[
    "GitHub"
].sum()

effect_size_rows = []

for question_type in QUESTION_TYPES:
    so_selected = counts.loc[
        "Stack Overflow",
        question_type,
    ]

    github_selected = counts.loc[
        "GitHub",
        question_type,
    ]

    so_other = (
        so_total - so_selected
    )

    github_other = (
        github_total - github_selected
    )

    (
        odds_ratio,
        lower_ci,
        upper_ci,
        p_value,
    ) = calculate_odds_ratio(
        so_selected,
        so_other,
        github_selected,
        github_other,
    )

    effect_size_rows.append(
        {
            "question_type":
                question_type.title(),
            "so_n":
                int(so_selected),
            "so_percent":
                100 * so_selected / so_total,
            "github_n":
                int(github_selected),
            "github_percent":
                100
                * github_selected
                / github_total,
            "difference_percentage_points":
                100
                * (
                    so_selected / so_total
                    - github_selected
                    / github_total
                ),
            "odds_ratio_so_vs_github":
                odds_ratio,
            "ci_95_lower":
                lower_ci,
            "ci_95_upper":
                upper_ci,
            "p_value":
                p_value,
        }
    )

effect_sizes = pd.DataFrame(
    effect_size_rows
)

effect_sizes["holm_adjusted_p"] = (
    holm_adjustment(
        effect_sizes["p_value"]
    )
)

effect_sizes[
    "significant_after_holm"
] = (
    effect_sizes["holm_adjusted_p"]
    < 0.05
)

print("\nQuestion-type effect sizes")
print(
    effect_sizes.round(4).to_string(
        index=False
    )
)

effect_sizes.to_csv(
    OUTPUT_DIR
    / "question_type_effect_sizes.csv",
    index=False,
)


# ============================================================
# MANUSCRIPT-READY SUMMARY
# ============================================================

if chi_square_p < 0.001:
    formatted_p = "p < 0.001"
else:
    formatted_p = (
        f"p = {chi_square_p:.3f}"
    )

summary = (
    "The distribution of question types differed significantly "
    "between Stack Overflow and GitHub, "
    f"chi-square({degrees_of_freedom}, "
    f"N = {sample_size:,}) = {chi_square:.2f}, "
    f"{formatted_p}, "
    f"Cramér's V = {cramers_v:.3f}."
)

print("\nManuscript-ready overall result")
print(summary)

with open(
    OUTPUT_DIR / "statistical_summary.txt",
    "w",
    encoding="utf-8",
) as output_file:
    output_file.write(summary + "\n")


# ============================================================
# LATEX TABLE
# ============================================================

latex_data = effect_sizes.copy()

latex_data[
    "Stack Overflow, n (\\%)"
] = latex_data.apply(
    lambda row: (
        f"{row['so_n']:,} "
        f"({row['so_percent']:.2f}\\%)"
    ),
    axis=1,
)

latex_data[
    "GitHub, n (\\%)"
] = latex_data.apply(
    lambda row: (
        f"{row['github_n']:,} "
        f"({row['github_percent']:.2f}\\%)"
    ),
    axis=1,
)

latex_data[
    "OR (95\\% CI)"
] = latex_data.apply(
    lambda row: (
        f"{row['odds_ratio_so_vs_github']:.2f} "
        f"({row['ci_95_lower']:.2f}--"
        f"{row['ci_95_upper']:.2f})"
    ),
    axis=1,
)

latex_data[
    "Adjusted $p$"
] = latex_data[
    "holm_adjusted_p"
].apply(
    lambda value: (
        "$<0.001$"
        if value < 0.001
        else f"{value:.3f}"
    )
)

latex_table = latex_data[
    [
        "question_type",
        "Stack Overflow, n (\\%)",
        "GitHub, n (\\%)",
        "OR (95\\% CI)",
        "Adjusted $p$",
    ]
].rename(
    columns={
        "question_type":
            "Question type"
    }
).to_latex(
    index=False,
    escape=False,
    caption=(
        "Cross-platform comparison of question types. "
        "Odds ratios greater than one indicate stronger "
        "association with Stack Overflow."
    ),
    label="tab:cross_platform_intent",
)

with open(
    OUTPUT_DIR
    / "cross_platform_table.tex",
    "w",
    encoding="utf-8",
) as latex_file:
    latex_file.write(latex_table)

print(
    "\nAnalysis completed. Results were saved in:",
    OUTPUT_DIR.resolve(),
)


Observed counts
question_type    how   why  what  other
platform                               
Stack Overflow   253    71   132     39
GitHub          2349  3995  1774    995

Platform percentages
question_type     how    why   what  other
platform                                  
Stack Overflow  51.11  14.34  26.67   7.88
GitHub          25.78  43.84  19.47  10.92

Overall cross-platform test
Chi-square: 224.0890
Degrees of freedom: 3
p-value: 2.6228471e-48
Cramér's V: 0.1527

Adjusted standardized residuals
question_type      how     why   what  other
platform                                    
Stack Overflow  12.353 -12.935  3.912 -2.125
GitHub         -12.353  12.935 -3.912  2.125

Question-type effect sizes
question_type  so_n  so_percent  github_n  github_percent  difference_percentage_points  odds_ratio_so_vs_github  ci_95_lower  ci_95_upper  p_value  holm_adjusted_p  significant_after_holm
          How   253     51.1111      2349         25.7764                       25.33